In [ ]:
import os
import itertools
import sys
from pathlib import Path

from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.filtering.log.attributes import attributes_filter
import pandas as pd
import warnings

%run classes_axillary.ipynb
#from classes_axillary import ColumnNames, ChangeMomentsInputFile

#from package.classes_axillary import ColumnNames, ChangeMomentsInputFile

warnings.filterwarnings('ignore')
from collections import defaultdict, Counter
from pm4py import write_xes
import numpy as np
from copy import deepcopy
from scipy.spatial.distance import cdist


def remove_outliers_iqr(data, outlier_range_multiplier=1.5):
    """
    Removes outliers above the Interquartile Range (IQR) from a list of floats.

    Args:
        data (list): List of float values.
        outlier_range_multiplier (float, optional): Multiplier for the IQR to define the outlier range.
            Default is 1.5.

    Returns:
        list: List with outliers removed.
    """
    
    # Ensure data is not empty
    if len(data) == 0:
        print("Warning: Input data is empty. Returning empty list.")
        return []
        #raise ValueError("Input data is empty. Cannot compute IQR.")

    # Calculate the first quartile (25th percentile)
    q1 = np.percentile(data, 25)

    # Calculate the third quartile (75th percentile)
    q3 = np.percentile(data, 75)

    # Calculate the interquartile range (IQR)
    iqr = q3 - q1

    # Define the range for identifying outliers
    outlier_range = outlier_range_multiplier * iqr

    # Create a new list with values within the outlier range
    filtered_data = [x for x in data if q1 - outlier_range <= x <= q3 + outlier_range]

    return filtered_data


def calculate_jaccard_similarity(dataframe):
    data_array = dataframe.values.astype(bool)
    similarity_matrix = cdist(data_array.T, data_array.T, metric='jaccard')
    similarity_df = pd.DataFrame(1 - similarity_matrix, columns=dataframe.columns, index=dataframe.columns)
    return similarity_df


def compare_two_sets(set1, set2):

    added = set2 - set1
    removed = set1 - set2

    return added, removed


def convert_dict_to_dataframe(dict):
    dict_df = pd.DataFrame.from_dict(dict)
    dict_df.fillna(0, inplace=True)

    return dict_df

def calculate_log_time_split(log, n_splits):

    timestamps_dict = attributes_filter.get_attribute_values(log, "time:timestamp")
    timestamps = list(timestamps_dict.keys())
    timestamps = [x.replace(tzinfo=None) for x in timestamps]
    step_in_seconds = np.ceil((max(timestamps) - min(timestamps)).total_seconds() / n_splits)
    start_time = min(timestamps).strftime("%Y-%m-%d %H:%M")
    timeframe_list = pd.period_range(start_time, periods=n_splits, freq=str(step_in_seconds)+'S')
    return list(timeframe_list.to_timestamp())



def load_event_log(path_to_log):
    variant = xes_importer.Variants.ITERPARSE
    parameters = {variant.value.Parameters.TIMESTAMP_SORT: True}
    event_log = xes_importer.apply(path_to_log, variant=variant, parameters=parameters)
    return event_log


# def load_event_log(path_to_log):
#
#     file_name = os.path.basename(os.path.normpath(path_to_log))
#     if file_name.endswith('.xes'):
#         variant = xes_importer.Variants.ITERPARSE
#         parameters = {variant.value.Parameters.TIMESTAMP_SORT: True}
#         event_log = xes_importer.apply(path_to_log, variant=variant, parameters=parameters)
#     else:
#         raise ValueError("Issue with event log loading")
#
#     return event_log



def event_log_preprocessing(log_initial):
    # Interval log
    # try:
    #     log_initial = attributes_filter.apply_events(log_initial, ["complete"], parameters={
    #         attributes_filter.Parameters.ATTRIBUTE_KEY: "lifecycle:transition",
    #         attributes_filter.Parameters.POSITIVE: True})
    # except:
    #     pass
    # Add additional attribute to the first and the last events in each trace
    log_preprocessed = add_event_attribute_start_end(log_initial)

    return log_preprocessed


def add_event_attribute_start_end(event_log):
    for trace in event_log:
        trace.insert(0, {'concept:name': 'start'})
        trace.append({'concept:name': 'end'})

    return event_log



def calc_timeframe_for_event_windows(file_name, log, list_of_change_moments):
    # transform dataframe to a list for a given event log
    event_windows = list(list_of_change_moments.loc[list_of_change_moments['log_name'] == file_name]['change_moment'])
    #event_windows = [x.to_pydatetime() for x in event_windows]

    timestamps_dict = attributes_filter.get_attribute_values(log, "time:timestamp")
    timestamps = list(timestamps_dict.keys())
    timestamps = [x.replace(tzinfo=None) for x in timestamps]

    event_windows.append(min(timestamps))
    event_windows.append(max(timestamps))
    event_windows.sort()
    #event_windows = [x.replace(tzinfo=pytz.UTC) for x in event_windows]

    return event_windows


def calc_behavioral_representation_per_window_full_traces(event_windows, log_initial):

    keys = [str(k) for k in range(1, len(event_windows))]
    behavioral_representation = defaultdict(list)
    for key in keys:
        behavioral_representation[key] = []
    #keys = {k: list for k in range(1, len(event_windows)+1)}
    #behavioral_representation = defaultdict(list, keys)
    #keys = [str(k) for k in range(1, len(event_windows)+1)]
    #behavioral_representation = dict.fromkeys(keys, [])

    for trace_id, trace in enumerate(log_initial):
        for event_id, event in enumerate(trace):
            if event_id > 0:
                directly_follows = (trace[event_id-1]['concept:name'], trace[event_id]['concept:name'])
                trace_start_event_time = trace[0]['time:timestamp']
                window_index = [i for i, t in enumerate(event_windows) if t > trace_start_event_time]
                if window_index:
                    behavioral_representation[str(window_index[0])].append(directly_follows)

    for window_id, directly_follows in behavioral_representation.items():
        behavioral_representation[window_id] = Counter(directly_follows)

    return behavioral_representation



def calc_directly_follows_between_two_timestamps(period, log_initial):

    behavioral_representation = defaultdict(list)

    for trace_id, trace in enumerate(log_initial):
        for event_id, event in enumerate(trace):
            if event_id > 0:
                directly_follows = (trace[event_id-1]['concept:name'], trace[event_id]['concept:name'])
                directly_follows_time = trace[event_id]['time:timestamp']
                window_index = [i for i, t in enumerate(period) if t >= directly_follows_time]
                if window_index:
                    behavioral_representation.append(directly_follows)

    behavioral_representation = Counter(behavioral_representation)

    return behavioral_representation


def get_batches_of_size_n(timeframe, n: int = 4):
    timeframe_batches = []

    for i in range(len(timeframe) - n + 1):
        batch = timeframe[i:i + n]
        timeframe_batches.append(batch)
        # TODO: split the middle window into two windows
        # w2_middle = batch[1] + (batch[2] - batch[1]) / 2
        # batch = [window_batch[0], window_batch[1], w2_middle, window_batch[2], window_batch[3]]
        # timeframe_batches.append(batch)

    return timeframe_batches


def add_additional_timestamp_to_batch_of_4_timestamps(timestamps_list):
    additional_timestamp = (timestamps_list[2] - timestamps_list[1]) / 2 + timestamps_list[1]
    timestamps_list.insert(2, additional_timestamp)
    return None




def extract_unique_sorted_integers(df, column_name):
    integer_list = []

    # Iterate over each element in the specified column
    for row_data in df[column_name]:
        row_integers = []
        for item in row_data:
            integers = item.strip('[]').split(',')
            for integer in integers:
                row_integers.append(int(integer))

        row_integers_unique = list(set(row_integers))
        row_integers_unique_sorted = sorted(row_integers_unique)
        integer_list.append(row_integers_unique_sorted)

    df['change_point'] = integer_list
    del df[column_name]

    return df




def get_all_paths_to_event_logs(input_dir, file_endwith: str = 'xes'):
    all_paths = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(file_endwith):
                all_paths.append(os.path.join(root, file))

    return all_paths


def create_eval_parameters(eval_config, results):
    log_name, complexity, noise_level = eval_config
    condition = (results.log_name == log_name) & (results.eval_noise_level == noise_level)

    if not results[condition].empty and not results[results.log_name == log_name].empty:

        method = results[condition].config_method.values[0]
        #window_size = results[condition].config_window_size.values[0]
        #opt_method = results[condition].config_opt_method.values[0]
        collection_name = results[results.log_name == log_name].collection_name.values[0]

        eval_parameters = {'log_name': log_name,
                           'complexity': complexity,
                           'noise_level': noise_level,
                           'method': method,
                           #'window_size': window_size,
                           #'opt_method': opt_method,
                           'collection_name': collection_name}
        return eval_parameters
    else:
        return None


def get_eval_parameter(path_to_log, parameters):

    parameters['log_name'] = os.path.basename(path_to_log)
    parameters['collection_name'] = path_to_log.split(os.sep)[-2]
    parameters['complexity'] = parameters['collection_name'].split('_')[-2]
    parameters['noise_level'] = get_noise_level_from_folder_name(path_to_log, 2)

    return parameters


def create_report(parameters, change_index, change_moment = None, change_type = None):
    result = pd.DataFrame()

    if len(change_index) == 0:
        change_index = ['na']
        change_moment = ['na']
        change_type = ['na']
    else:
        if change_moment is None:
            change_moment = ['na'] * len(change_index)
        if change_type is None:
            change_type = ['na'] * len(change_index)

    detected_info = list(zip(change_index, change_moment, change_type))
    for change_id, change_info in enumerate(detected_info):
        change_index_instance, change_moment_instance, change_type_instance = change_info
        record = {
            'collection_name': parameters['collection_name'],
            'log_name': parameters['log_name'],
            'log_size': parameters['log_size'],
            'eval_noise_level': parameters['noise_level'],
            'eval_complexity': parameters['complexity'],
            'calc_change_id': change_id+1,
            'calc_change_index': change_index_instance,
            'calc_change_moment': change_moment_instance,
            'calc_change_type': change_type_instance,
            'calc_drift_type': 'simple',
            'config_method': parameters['method'],
            'config_window_size': parameters['window_size'],
            }
        result = pd.concat([result, pd.DataFrame.from_records([record])], ignore_index=True)

    return result


def get_noise_level_from_folder_name(path_to_log, position: int = 2):
    noise_setting = path_to_log.split(os.sep)[-position]

    if noise_setting == 'without_noise':
        noise_level = 0
    elif noise_setting == 'with_noise_10':
        noise_level = 10
    elif noise_setting == 'with_noise_20':
        noise_level = 20
    else:
        sys.exit()
    return noise_level


def load_and_preprocess_drift_info(path_to_file):
    drift_info = pd.read_csv(path_to_file)
    drift_info_adjusted = gold_standard_preprocessing(drift_info)

    return drift_info_adjusted


def gold_standard_preprocessing(flat_file):

    # Extract the main data related to timestamps
    gold_standard_adjusted = flat_file.loc[(flat_file['drift_sub_attribute'].isin(['change_start', 'change_end']))]
    # Extract drift and change types
    drift_types = flat_file.loc[flat_file['drift_attribute'].isin(['drift_type'])]
    change_types = flat_file.loc[(flat_file['drift_sub_attribute'].isin(['change_type']))]
    change_index = flat_file.loc[(flat_file['drift_sub_attribute'].isin(['change_trace_index']))]

    # Add drift and change types tot the main data
    gold_standard_adjusted = pd.merge(gold_standard_adjusted,
                              drift_types[['log_name', 'drift_or_noise_id', 'value']],
                              on=['log_name', 'drift_or_noise_id'],
                              how='left')
    gold_standard_adjusted = pd.merge(gold_standard_adjusted,
                              change_types[['log_name', 'drift_or_noise_id', 'drift_attribute', 'value']],
                              on=['log_name', 'drift_or_noise_id', 'drift_attribute'],
                              how='left')

    gold_standard_adjusted.rename(columns={'value': ChangeMomentsInputFile.change_type.value,
                                   'value_y': ChangeMomentsInputFile.drift_type.value,
                                   'value_x': ChangeMomentsInputFile.timestamp.value}, inplace=True)

    gold_standard_adjusted = pd.merge(gold_standard_adjusted,
                              change_index[['log_name', 'drift_or_noise_id', 'drift_attribute', 'value']],
                              on=['log_name', 'drift_or_noise_id', 'drift_attribute'],
                              how='left')
    # Rename and reorder main data
    gold_standard_adjusted.rename(columns={'value': ChangeMomentsInputFile.change_index.value,
                                   'drift_or_noise_id': ChangeMomentsInputFile.drift_id.value,
                                   'drift_attribute': ChangeMomentsInputFile.change_id.value,
                                   'drift_sub_attribute': ChangeMomentsInputFile.change_start_end.value},
                          inplace=True)

    column_order = [ChangeMomentsInputFile.complexity.value,
                    ChangeMomentsInputFile.log_name.value,
                    ChangeMomentsInputFile.drift_id.value,
                    ChangeMomentsInputFile.drift_type.value,
                    ChangeMomentsInputFile.change_id.value,
                    ChangeMomentsInputFile.change_type.value,
                    ChangeMomentsInputFile.timestamp.value,
                    ChangeMomentsInputFile.change_index.value,
                    ChangeMomentsInputFile.change_start_end.value]

    gold_standard_adjusted = gold_standard_adjusted.reindex(column_order, axis=1)
    # Drop duplicates since sudden change types do not have and end timestamp
    gold_standard_adjusted.drop_duplicates(subset=column_order[:-2], inplace=True)

    gold_standard_adjusted.loc[(gold_standard_adjusted[ChangeMomentsInputFile.change_type.value] == 'gradual') &
                       (gold_standard_adjusted[
                            ChangeMomentsInputFile.change_start_end.value] == 'change_start'), ChangeMomentsInputFile.change_type.value] = 'gradual_start'
    gold_standard_adjusted.loc[(gold_standard_adjusted[ChangeMomentsInputFile.change_type.value] == 'gradual') &
                       (gold_standard_adjusted[
                            ChangeMomentsInputFile.change_start_end.value] == 'change_end'), ChangeMomentsInputFile.change_type.value] = 'gradual_end'

    return gold_standard_adjusted


def create_and_save_gold_standard_from_drift_info(path_to_drift_info, save: bool = False):
    gold_standard = load_and_preprocess_drift_info(path_to_drift_info)
    grouped_info = gold_standard.groupby(['log_name']).agg({'drift_id': list,
                                                            'change_index': list,
                                                            'change_type': list,
                                                            'drift_type': list}).reset_index()

    grouped_info = pd.merge(grouped_info, gold_standard[['log_name', 'complexity']], how='left', on=['log_name'])

    gold_standard = extract_unique_sorted_integers(grouped_info, 'change_index')
    gold_standard = gold_standard.drop_duplicates(subset=['log_name'])

    data_collection_info = pd.read_csv(path_to_drift_info.parent / 'data_collection_analysis.csv')
    gold_standard = pd.merge(gold_standard, data_collection_info[['log_name', 'log_size']], how='left', on = ['log_name'])

    if save:
        gold_standard.to_csv(Path(path_to_drift_info.parent, 'gold_standard.csv'), index=False)

    return gold_standard


def create_configuration_set(gold_standard, results):
    complexity = gold_standard.complexity.tolist()
    log_names = gold_standard.log_name.tolist()
    noise_levels = list(set(results.eval_noise_level.tolist()))
    log_names_in_scope = results[['eval_noise_level', 'log_name']].drop_duplicates().values.tolist()
    log_names_joined = [name for name in log_names if name in [n[-1] for n in log_names_in_scope]]
    log_name_complexity = list(set(zip(log_names_joined, complexity)))
    eval_configs = list(itertools.product(log_name_complexity, noise_levels))
    eval_configs = [(*item[0], item[1]) for item in eval_configs]
    return eval_configs
